<a href="https://colab.research.google.com/github/schnekenberg/tcc-burned-area-detection/blob/main/02_data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y gdal-bin

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  python3-gdal python3-numpy
Suggested packages:
  libgdal-grass python-numpy-doc python3-dev python3-pytest
The following NEW packages will be installed:
  gdal-bin python3-gdal python3-numpy
0 upgraded, 3 newly installed, 0 to remove and 2 not upgraded.
Need to get 5,168 kB of archives.
After this operation, 25.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-numpy amd64 1:1.21.5-1ubuntu22.04.1 [3,467 kB]
Get:2 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 python3-gdal amd64 3.8.4+dfsg-1~jammy0 [1,095 kB]
Get:3 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 gdal-bin amd64 3.8.4+dfsg-1~jammy0 [605 kB]
Fetched 5,168 kB in 4s (1,351 kB/s)
Selecting previously unselected package python3-numpy.
(Reading database ... 118242 file

In [ ]:
import os
import subprocess
import glob
import numpy as np
import math
import rasterio
from rasterio.windows import Window
from rasterio.windows import bounds as window_bounds
from rasterio.transform import array_bounds
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive
from google.colab import auth

auth.authenticate_user()

result = subprocess.run(
    ['gcloud', 'auth', 'print-access-token'],
    capture_output=True, text=True
)
token = result.stdout.strip()

os.environ['GDAL_HTTP_HEADER_FILE'] = ''
os.environ['GCS_OAUTH2_TOKEN'] = token

drive.mount('/content/drive')

GCS_BUCKET = 'gs://tcc-roraima-dataset'
CSV_FOLDER = '/content/drive/MyDrive/CSV_files'
LOCAL_DIR  = '/content/rasters'  # precisaremos carregar cada arquivo localmente para processamento
os.makedirs(LOCAL_DIR, exist_ok = True)

FOREST_THRESHOLD = 0.5
TILE_SIZE = 128
SEED = 42
BANDS = ['B2','B3','B4','B8','B12','NDVI','NBR']
FEATURE_COLS = [f'{stat}_{band}' for stat in ['mean','std'] for band in BANDS]

MONTHS = ['202401', '202402', '202403', '202404', '202411', '202412']

def build_vrt_local(output_name, pattern):
    tifs = sorted(glob.glob(f"{LOCAL_DIR}/{pattern}*.tif"))
    out  = f"{LOCAL_DIR}/{output_name}.vrt"
    if tifs:
        subprocess.run(["gdalbuildvrt", out] + tifs, check = True)
        print(f"{output_name}.vrt criado com {len(tifs)} arquivos")
    else:
        print(f"Nenhum .TIF encontrado para: {pattern}")
    return out

# confirma acesso ao bucket ao listar os arquivos
!gsutil ls {GCS_BUCKET}

Mounted at /content/drive
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://tcc-roraima-dataset/roraima_composite_2024010000000000-0000000000.tif
gs://tcc-roraima-dataset/roraima_composite_2024010000000000-0000012544.tif
gs://tcc-roraima-dataset/roraima_composite_2024010000000000-0000025088.tif
gs://tcc-roraima-dataset/roraima_composite_2024010000000000-0000037632.tif
gs://tcc-roraima-dataset/roraima_composite_2024010000000000-0000050176.tif
gs://tcc-roraima-dataset/roraima_composite_2024010000000000-0000062720.tif
gs://tcc-roraima-dataset/roraima_composite_2024010000012544-0000000000.tif
gs://tcc-roraima-dataset/roraima_composite_2024010000012544-0000012544.tif
gs://tcc-roraima-dataset/roraima_composite_2024010000012544-0000025088.tif
gs://tcc-roraima-dataset/roraima_composi

In [ ]:
# baixa máscaras estáticas (apenas uma vez) do GCS: floresta e TIs
os.system(f"gsutil -m cp 'gs://tcc-roraima-dataset/roraima_forest_mask*.tif' {LOCAL_DIR}/")
os.system(f"gsutil -m cp 'gs://tcc-roraima-dataset/roraima_indigenous_mask*.tif' {LOCAL_DIR}/")

# monta .VRTs localmente
build_vrt_local("roraima_forest_mask", "roraima_forest_mask")
build_vrt_local("roraima_indigenous_mask", "roraima_indigenous_mask")

forest_src = rasterio.open(f"{LOCAL_DIR}/roraima_forest_mask.vrt")
ti_src = rasterio.open(f"{LOCAL_DIR}/roraima_indigenous_mask.vrt")

# 1. aplica filtro de floresta para coletar tiles válidos, a partir da máscara de floresta
width = forest_src.width
height = forest_src.height
rows = math.floor(height / TILE_SIZE)
cols = math.floor(width / TILE_SIZE)

tiles_validos = []

for r in range(rows):
    for c in range(cols): # percorre cada tile da máscara de floresta e mede a % de floresta

        win = Window(c * TILE_SIZE, r * TILE_SIZE, TILE_SIZE, TILE_SIZE)
        forest_window = forest_src.read(1, window = win) # pega pixels onde a máscara de floresta = 1

        if forest_src.nodata is not None:
            if np.all(forest_window == forest_src.nodata):
                continue # descarta

        forest_fraction = forest_window.mean() # média
        if forest_fraction < FOREST_THRESHOLD: # proporção de pixels = 1 na máscara é maior que o threshold?
            continue # descarta

        tiles_validos.append((win, r, c)) # tile válido: extrai chip da imagem

print(f"Tiles válidos: {len(tiles_validos)}")

roraima_forest_mask.vrt criado com 4 arquivos
roraima_indigenous_mask.vrt criado com 4 arquivos
Tiles válidos: 106309


In [ ]:
# 2. rotulação com base na máscara de fogo
all_records = []

for month in MONTHS:

  print(f"\nProcessando mês {month}...")

  # baixa arquivos do mês do GCS
  os.system(f"gsutil -m cp 'gs://tcc-roraima-dataset/roraima_composite_{month}*.tif' {LOCAL_DIR}/")
  os.system(f"gsutil -m cp 'gs://tcc-roraima-dataset/roraima_fire_mask_{month}*.tif' {LOCAL_DIR}/")

  # monta .VRTs localmente
  build_vrt_local(f"roraima_composite_{month}", f"roraima_composite_{month}")
  build_vrt_local(f"roraima_fire_mask_{month}", f"roraima_fire_mask_{month}")

  composite_src = rasterio.open(f"{LOCAL_DIR}/roraima_composite_{month}.vrt")
  fire_src = rasterio.open(f"{LOCAL_DIR}/roraima_fire_mask_{month}.vrt")

  for win, r, c in tqdm(tiles_validos, desc = f"Rotulando tiles de {month}"):
      # coordenadas geográficas do tile
      left, bottom, right, top = window_bounds(win, composite_src.transform)
      centroid_lon = (left + right) / 2
      centroid_lat = (bottom + top) / 2

      # rótulo: qualquer pixel de fogo no tile = 1 na máscara de fogo
      fire_chip = fire_src.read(1, window = win)
      label = int(np.any(fire_chip > 0))

      # feature is_indigenous: tile intersecta TI?
      ti_chip = ti_src.read(1, window = win)
      is_indigenous = int(np.any(ti_chip > 0))

      # features: calcula média e desvio padrão por banda espectral
      image_chip = composite_src.read(window = win)
      row_data = { # salva rótulo e coordenadas
          'month': month,
          'label': label,
          'is_indigenous': is_indigenous,
          'tile_row': r,
          'tile_col': c,
          'centroid_lon': centroid_lon,
          'centroid_lat': centroid_lat,
      }
      for i, band in enumerate(BANDS):
          row_data[f'mean_{band}'] = image_chip[i].mean()
          row_data[f'std_{band}'] = image_chip[i].std()

      all_records.append(row_data)

  composite_src.close()
  fire_src.close()

  # é necessário deletar os arquivos locais após serem processados para que haja espaço de armazenamento para o próximo mês
  for f in glob.glob(f"{LOCAL_DIR}/roraima_composite_{month}*.tif"):
        os.remove(f)
  for f in glob.glob(f"{LOCAL_DIR}/roraima_fire_mask_{month}*.tif"):
        os.remove(f)

  os.remove(f"{LOCAL_DIR}/roraima_composite_{month}.vrt")
  os.remove(f"{LOCAL_DIR}/roraima_fire_mask_{month}.vrt")
  print(f"Mês {month} concluído. Seus arquivos foram deletados")

# tratamento de dados NaN
df = pd.DataFrame(all_records)
antes = len(df)
df = df.dropna(subset = FEATURE_COLS)
depois = len(df)

print(f"Tiles removidos por NaN: {antes - depois}")
print(f"Com fogo (1): {df['label'].sum()}")
print(f"Sem fogo (0): {(df['label'] == 0).sum()}")
print(f"Dentro de TI (1): {df['is_indigenous'].sum()}")
print(f"Fora de TI (0): {(df['is_indigenous'] == 0).sum()}")



Processando mês 202401...
roraima_composite_202401.vrt criado com 42 arquivos
roraima_fire_mask_202401.vrt criado com 4 arquivos


Rotulando tiles de 202401: 100%|██████████| 106309/106309 [12:03<00:00, 146.89it/s]


Mês 202401 concluído. Seus arquivos foram deletados

Processando mês 202402...
roraima_composite_202402.vrt criado com 42 arquivos
roraima_fire_mask_202402.vrt criado com 4 arquivos


Rotulando tiles de 202402: 100%|██████████| 106309/106309 [12:29<00:00, 141.78it/s]


Mês 202402 concluído. Seus arquivos foram deletados

Processando mês 202403...
roraima_composite_202403.vrt criado com 42 arquivos
roraima_fire_mask_202403.vrt criado com 4 arquivos


...

Mês 202404 concluído. Seus arquivos foram deletados

Processando mês 202411...
roraima_composite_202411.vrt criado com 42 arquivos
roraima_fire_mask_202411.vrt criado com 4 arquivos


Rotulando tiles de 202411: 100%|██████████| 106309/106309 [11:25<00:00, 155.09it/s]


Mês 202411 concluído. Seus arquivos foram deletados

Processando mês 202412...
roraima_composite_202412.vrt criado com 42 arquivos
roraima_fire_mask_202412.vrt criado com 4 arquivos


Rotulando tiles de 202412: 100%|██████████| 106309/106309 [09:43<00:00, 182.31it/s]


Mês 202412 concluído. Seus arquivos foram deletados
Tiles removidos por NaN: 203414
Com fogo (1): 63721
Sem fogo (0): 370719
Dentro de TI (1): 243713
Fora de TI (0): 190727


In [ ]:
# salva o dataframe consolidado no drive antes do split para não correr o risco de perdê-lo
df.to_csv(f"{CSV_FOLDER}/dataset_completo.csv", index = False)
print(f"Dataset salvo: {len(df)} registros")

Dataset salvo: 434440 registros


In [ ]:
# 3. separar datasets (70-15-15)
# abre o dataframe salvo para evitar problemas de sessão colab expirada
df = pd.read_csv(f"{CSV_FOLDER}/dataset_completo.csv")

# primeiro split: 70% treino, 30% restante
train_df, temp_df = train_test_split(
    df,
    test_size = 0.30,
    random_state = SEED,
    stratify = df['label']
)

# segundo split: 15% validação, 15% teste
val_df, test_df = train_test_split(
    temp_df,
    test_size = 0.50,
    random_state = SEED,
    stratify = temp_df['label']
)

print(f"\nTreino: {len(train_df)} tiles | fogo: {train_df['label'].sum()}")
print(f"\nValidação: {len(val_df)} tiles | fogo: {val_df['label'].sum()}")
print(f"\nTeste: {len(test_df)} tiles | fogo: {test_df['label'].sum()}")

train_df.to_csv(f"{CSV_FOLDER}/train.csv", index = False)
val_df.to_csv(f"{CSV_FOLDER}/val.csv", index = False)
test_df.to_csv(f"{CSV_FOLDER}/test.csv", index = False)



Treino: 304108 tiles | fogo: 44605

Validação: 65166 tiles | fogo: 9558

Teste: 65166 tiles | fogo: 9558


In [ ]:
print(f"Treino: {len(train_df)}")
print(f"Validação: {len(val_df)}")
print(f"Teste: {len(test_df)}")
print(f"Val == Test: {val_df.equals(test_df)}")

# confirma que não há overlap real nos dados
overlap = pd.merge(val_df, test_df, how='inner')
print(f"Registros em comum: {len(overlap)}")

Treino: 304108
Validação: 65166
Teste: 65166
Val == Test: False
Registros em comum: 0
